# Step 2: Create PPD Plan-Level Raw Data

Replicates `code/create_ppd_dta.do`

**Inputs:**
- `raw/us_census_bureau_regions_and_divisions.csv` (Census region lookup)
- `raw/ppd-data-2023-07-28.csv` (main PPD data)
- `output/mapping_table_fy_final.dta` (Step 1 mapping table — Stata baseline)

**Output:** `output_python/ppd_plan_level_raw.{dta,parquet,csv}`  
**Parity baseline:** `output/ppd_plan_level_raw.dta`

In [1]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT = Path.cwd().parent  # assumes notebook runs from notebooks/
RAW = ROOT / 'raw'
OUTPUT_STATA = ROOT / 'output'
OUTPUT_PY = ROOT / 'output_python'
OUTPUT_PY.mkdir(exist_ok=True)

print(f'Root: {ROOT}')
print(f'Raw:  {RAW}')
print(f'Stata baseline: {OUTPUT_STATA}')
print(f'Python output:  {OUTPUT_PY}')

Root: /Users/Work/Desktop/Pension Research/ppd-data
Raw:  /Users/Work/Desktop/Pension Research/ppd-data/raw
Stata baseline: /Users/Work/Desktop/Pension Research/ppd-data/output
Python output:  /Users/Work/Desktop/Pension Research/ppd-data/output_python


## 1. Import Census Region Lookup

Stata imports without `case(preserve)`, so column names are lowercased and spaces removed.  
Then: `rename statecode StateAbbrev` and `drop state`.

In [2]:
regions = pd.read_csv(RAW / 'us_census_bureau_regions_and_divisions.csv')
print(f'Region CSV columns (raw): {list(regions.columns)}')

# Mimic Stata: import delimited lowercases names and removes spaces
regions.columns = regions.columns.str.lower().str.replace(' ', '', regex=False)

# rename statecode -> StateAbbrev, drop state
regions = regions.rename(columns={'statecode': 'StateAbbrev'})
regions = regions.drop(columns=['state'])

print(f'Region lookup columns: {list(regions.columns)}')
print(f'Region lookup rows: {len(regions)}')
regions.head()

Region CSV columns (raw): ['State', 'State Code', 'Region', 'Division']
Region lookup columns: ['StateAbbrev', 'region', 'division']
Region lookup rows: 51


,StateAbbrev,region,division
0,AK,West,Pacific
1,AL,South,East South Central
2,AR,South,West South Central
3,AZ,West,Mountain
4,CA,West,Pacific


## 2. Import PPD CSV

Stata uses `case(preserve)` so column names retain original casing.  
Then drop rows where `fy` is missing.

In [4]:
ppd = pd.read_csv(RAW / 'ppd-data-2023-07-28.csv', encoding='cp1252')
print(f'PPD raw: {ppd.shape[0]} rows x {ppd.shape[1]} cols')
print(f'Missing fy: {ppd["fy"].isna().sum()}')

# drop if mi(fy)
ppd = ppd.dropna(subset=['fy'])
ppd['fy'] = ppd['fy'].astype(int)
print(f'After drop mi(fy): {ppd.shape[0]} rows x {ppd.shape[1]} cols')

PPD raw: 6269 rows x 268 cols
Missing fy: 1
After drop mi(fy): 6268 rows x 268 cols


## 3. Merge 1 — Mapping Table (m:1 on ppd_id, fy)

Stata: `merge m:1 ppd_id fy using mapping_table_fy_final.dta, keep(1 3) nogen`  
`keep(1 3)` = left join (keep master-only + matched, drop using-only).

In [5]:
# Load mapping table (Stata baseline for isolation)
mapping = pd.read_stata(OUTPUT_STATA / 'mapping_table_fy_final.dta')
print(f'Mapping table: {mapping.shape[0]} rows x {mapping.shape[1]} cols')
print(f'Mapping columns: {list(mapping.columns)}')

# Ensure fy types align for merge
mapping['fy'] = mapping['fy'].astype(int)
mapping['ppd_id'] = mapping['ppd_id'].astype(int)
ppd['ppd_id'] = ppd['ppd_id'].astype(int)

# Left merge = keep(1 3)
n_before = len(ppd)
ppd = ppd.merge(mapping, on=['ppd_id', 'fy'], how='left', indicator=True)

# Diagnostics
merge_counts = ppd['_merge'].value_counts()
print(f'\nMapping merge diagnostics:')
print(f'  Matched (both):     {merge_counts.get("both", 0)}')
print(f'  Left-only (master): {merge_counts.get("left_only", 0)}')
print(f'  Rows before: {n_before}, after: {len(ppd)}')

ppd = ppd.drop(columns=['_merge'])

Mapping table: 5244 rows x 15 cols
Mapping columns: ['fy', 'ppd_id', 'ppd_name', 'pub_id', 'pub_system_name', 'preqin_id', 'preqin_name', 'tr13fid1', 'tr13fname1', 'tr13fid2', 'tr13fname2', 'tr13fid3', 'tr13fname3', 'tr13fid4', 'tr13fname4']

Mapping merge diagnostics:
  Matched (both):     5016
  Left-only (master): 1252
  Rows before: 6268, after: 6268


## 4. Merge 2 — Census Regions (m:1 on StateAbbrev)

Stata: `merge m:1 StateAbbrev using region, keep(1 3) nogen`

In [6]:
n_before = len(ppd)
ppd = ppd.merge(regions, on='StateAbbrev', how='left', indicator=True)

merge_counts = ppd['_merge'].value_counts()
print(f'Region merge diagnostics:')
print(f'  Matched (both):     {merge_counts.get("both", 0)}')
print(f'  Left-only (master): {merge_counts.get("left_only", 0)}')
print(f'  Rows before: {n_before}, after: {len(ppd)}')

ppd = ppd.drop(columns=['_merge'])

Region merge diagnostics:
  Matched (both):     6268
  Left-only (master): 0
  Rows before: 6268, after: 6268


## 5. Sort and Save

Stata: `sort ppd_id fy` then `save ppd_plan_level_raw.dta, replace`

In [7]:
ppd = ppd.sort_values(['ppd_id', 'fy']).reset_index(drop=True)
print(f'Final shape: {ppd.shape[0]} rows x {ppd.shape[1]} cols')
print(f'First 5 rows:')
ppd.head()

Final shape: 6268 rows x 283 cols
First 5 rows:


,ppd_id,PlanName,fy,system_id,PlanFullName,source_PlanBasics,FiscalYearType,PlanInceptionYear,PlanClosed,PlanYearClosed,...,tr13fid1,tr13fname1,tr13fid2,tr13fname2,tr13fid3,tr13fname3,tr13fid4,tr13fname4,region,division
0,1,Alabama ERS,2001,1.0,Employees’ Retirement System of Alabama,NaN,2.0,1945.0,0.0,NaN,...,9059.0,Retirement Systems Of Alabama,NaN,,NaN,,NaN,,South,East South Central
1,1,Alabama ERS,2002,1.0,Employees’ Retirement System of Alabama,NaN,2.0,1945.0,0.0,NaN,...,NaN,,NaN,,NaN,,NaN,,South,East South Central
2,1,Alabama ERS,2003,1.0,Employees’ Retirement System of Alabama,NaN,2.0,1945.0,0.0,NaN,...,NaN,,NaN,,NaN,,NaN,,South,East South Central
3,1,Alabama ERS,2004,1.0,Employees’ Retirement System of Alabama,NaN,2.0,1945.0,0.0,NaN,...,NaN,,NaN,,NaN,,NaN,,South,East South Central
4,1,Alabama ERS,2005,1.0,Employees’ Retirement System of Alabama,NaN,2.0,1945.0,0.0,NaN,...,NaN,,NaN,,NaN,,NaN,,South,East South Central


In [8]:
# Save DTA
pyreadstat.write_dta(
    ppd,
    OUTPUT_PY / 'ppd_plan_level_raw.dta'
)

# Save Parquet
ppd.to_parquet(OUTPUT_PY / 'ppd_plan_level_raw.parquet', index=False)

# Save CSV
ppd.to_csv(OUTPUT_PY / 'ppd_plan_level_raw.csv', index=False)

print('Saved: ppd_plan_level_raw.{dta, parquet, csv}')

Saved: ppd_plan_level_raw.{dta, parquet, csv}


## 6. Parity Check vs Stata Baseline

Compare against `output/ppd_plan_level_raw.dta`: row counts, column sets, null profiles, and value-level comparison.

In [13]:
# Load Stata baseline
stata_df = pd.read_stata(OUTPUT_STATA / 'ppd_plan_level_raw.dta')
py_df = ppd.copy()

print('=== PARITY CHECK: ppd_plan_level_raw ===')
results = []

# 1. Row count
row_match = len(py_df) == len(stata_df)
results.append(('Row count', row_match))
print(f'\n1. Row count: Python={len(py_df)}, Stata={len(stata_df)} -> {"PASS" if row_match else "FAIL"}')

# 2. Column set
py_cols = set(py_df.columns)
stata_cols = set(stata_df.columns)
col_match = py_cols == stata_cols
results.append(('Column set', col_match))
print(f'\n2. Column set match: {"PASS" if col_match else "FAIL"}')
if not col_match:
    only_py = py_cols - stata_cols
    only_stata = stata_cols - py_cols
    if only_py:
        print(f'   Only in Python: {sorted(only_py)}')
    if only_stata:
        print(f'   Only in Stata:  {sorted(only_stata)}')

# 3. Column order
col_order_match = list(py_df.columns) == list(stata_df.columns)
results.append(('Column order', col_order_match))
print(f'\n3. Column order match: {"PASS" if col_order_match else "FAIL"}')
if not col_order_match:
    # Show first divergence
    for i, (pc, sc) in enumerate(zip(py_df.columns, stata_df.columns)):
        if pc != sc:
            print(f'   First mismatch at position {i}: Python="{pc}" vs Stata="{sc}"')
            break

# 4. Key uniqueness
py_dups = py_df.duplicated(subset=['ppd_id', 'fy'], keep=False).sum()
stata_dups = stata_df.duplicated(subset=['ppd_id', 'fy'], keep=False).sum()
dup_match = py_dups == stata_dups
results.append(('Key duplicates', dup_match))
print(f'\n4. Duplicate rows on (ppd_id, fy): Python={py_dups}, Stata={stata_dups} -> {"PASS" if dup_match else "FAIL"}')

# 5. Null profile for key columns
key_cols = ['fy', 'ppd_id', 'StateAbbrev']
# Add merged columns if they exist in both
for c in ['pub_id', 'pub_system_name', 'region', 'division', 'Region', 'Division']:
    if c in py_df.columns and c in stata_df.columns:
        key_cols.append(c)
null_ok = True
print(f'\n5. Null profiles for key columns:')
for c in key_cols:
    if c in py_df.columns and c in stata_df.columns:
        pn = py_df[c].isna().sum()
        sn = stata_df[c].isna().sum()
        # Also count empty strings as missing (handles both object and StringDtype)
        if pd.api.types.is_string_dtype(stata_df[c]):
            sn += (stata_df[c] == '').sum()
        if pd.api.types.is_string_dtype(py_df[c]):
            pn += (py_df[c] == '').sum()
        match = pn == sn
        if not match:
            null_ok = False
        print(f'   {c}: Python null={pn}, Stata null={sn} -> {"PASS" if match else "FAIL"}')
results.append(('Null profiles', null_ok))

# Summary
all_pass = all(r[1] for r in results)
print(f'\n{"=" * 50}')
for name, passed in results:
    print(f'  {"PASS" if passed else "FAIL"} - {name}')
print(f'{"=" * 50}')
print(f'Overall: {"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}')

=== PARITY CHECK: ppd_plan_level_raw ===

1. Row count: Python=6268, Stata=6268 -> PASS

2. Column set match: PASS

3. Column order match: PASS

4. Duplicate rows on (ppd_id, fy): Python=0, Stata=0 -> PASS

5. Null profiles for key columns:
   fy: Python null=0, Stata null=0 -> PASS
   ppd_id: Python null=0, Stata null=0 -> PASS
   StateAbbrev: Python null=0, Stata null=0 -> PASS
   pub_id: Python null=1252, Stata null=1252 -> PASS
   pub_system_name: Python null=1252, Stata null=1252 -> PASS
   region: Python null=0, Stata null=0 -> PASS
   division: Python null=0, Stata null=0 -> PASS

  PASS - Row count
  PASS - Column set
  PASS - Column order
  PASS - Key duplicates
  PASS - Null profiles
Overall: ALL CHECKS PASSED


## 7. Deep Parity — Cell-by-Cell Value Comparison

In [18]:
# Align both DataFrames by canonical sort on ppd_id, fy
common_cols = sorted(set(py_df.columns) & set(stata_df.columns))

py_norm = py_df[common_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
stata_norm = stata_df[common_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)

# Normalize empty strings -> NaN for string columns (handles StringDtype)
for c in common_cols:
    if pd.api.types.is_string_dtype(stata_norm[c]):
        stata_norm[c] = stata_norm[c].replace('', np.nan)
    if pd.api.types.is_string_dtype(py_norm[c]):
        py_norm[c] = py_norm[c].replace('', np.nan)

# Compare column by column
mismatches = {}
for col in common_cols:
    p = py_norm[col]
    s = stata_norm[col]

    # Both NaN = match
    both_nan = p.isna() & s.isna()
    both_valid = p.notna() & s.notna()

    # NaN mismatch
    nan_mismatch = (p.isna() != s.isna()).sum()

    # Value mismatch among non-null
    # Use relative tolerance to handle Stata float32 vs Python float64
    if both_valid.any():
        p_dtype = str(p.dtype)
        s_dtype = str(s.dtype)
        if 'float' in p_dtype or 'float' in s_dtype or 'int' in p_dtype or 'int' in s_dtype:
            try:
                pv = pd.to_numeric(p[both_valid], errors='coerce')
                sv = pd.to_numeric(s[both_valid], errors='coerce')
                val_mismatch = int((~np.isclose(pv, sv, rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
            except Exception:
                val_mismatch = (p[both_valid].astype(str) != s[both_valid].astype(str)).sum()
        else:
            val_mismatch = (p[both_valid].astype(str) != s[both_valid].astype(str)).sum()
    else:
        val_mismatch = 0

    total = nan_mismatch + val_mismatch
    if total > 0:
        mismatches[col] = {'nan_diff': int(nan_mismatch), 'val_diff': int(val_mismatch), 'total': int(total)}

if mismatches:
    print(f'MISMATCHES found in {len(mismatches)} column(s):')
    for col, info in sorted(mismatches.items()):
        print(f'  {col}: NaN diff={info["nan_diff"]}, value diff={info["val_diff"]}, total={info["total"]}')
else:
    print(f'All {len(common_cols)} common columns match perfectly across {len(py_norm)} rows.')

# Columns not in common
only_py = sorted(set(py_df.columns) - set(stata_df.columns))
only_stata = sorted(set(stata_df.columns) - set(py_df.columns))
if only_py:
    print(f'\nColumns only in Python ({len(only_py)}): {only_py}')
if only_stata:
    print(f'\nColumns only in Stata ({len(only_stata)}): {only_stata}')

overall = len(mismatches) == 0 and len(only_py) == 0 and len(only_stata) == 0
print(f'\n{"EXACT PARITY CONFIRMED" if overall else "PARITY ISSUES DETECTED"}')

All 283 common columns match perfectly across 6268 rows.

EXACT PARITY CONFIRMED


In [20]:
# Final summary
n_common = len(common_cols)
n_mm = len(mismatches)
only_py_f = sorted(set(py_df.columns) - set(stata_df.columns))
only_stata_f = sorted(set(stata_df.columns) - set(py_df.columns))
exact = n_mm == 0 and len(only_py_f) == 0 and len(only_stata_f) == 0

print(f'Common columns: {n_common}')
print(f'Mismatched columns: {n_mm}')
print(f'Columns only in Python: {len(only_py_f)}')
print(f'Columns only in Stata: {len(only_stata_f)}')
print(f'Rows compared: {len(py_norm)}')
print(f'\n{"✅ EXACT PARITY CONFIRMED" if exact else "❌ PARITY ISSUES"}')

Common columns: 283
Mismatched columns: 0
Columns only in Python: 0
Columns only in Stata: 0
Rows compared: 6268

✅ EXACT PARITY CONFIRMED


## 8. Comprehensive Audit

Thorough multi-dimensional audit to ensure Python output matches `output/ppd_plan_level_raw.dta` exactly.

**Checks:**
1. Shape parity (rows × columns)
2. Column names and ordering (exact match)
3. Column dtype alignment (numeric vs string categories)
4. Full null profile across ALL columns
5. Spot-check specific rows by ppd_id+fy against Stata
6. Verify region column naming matches Stata (`region`/`division` vs `Region`/`Division`)
7. Merge diagnostics: confirm keep(1 3) semantics preserved
8. Exhaustive cell-by-cell value comparison with tolerance breakdown

In [21]:
print('=' * 60)
print('        COMPREHENSIVE STEP 2 AUDIT')
print('=' * 60)

# Reload fresh from saved files to ensure we audit what was actually written
py_audit = pd.read_stata(OUTPUT_PY / 'ppd_plan_level_raw.dta')
stata_audit = pd.read_stata(OUTPUT_STATA / 'ppd_plan_level_raw.dta')

audit_results = []

# ------------------------------------------------------------------
# CHECK 1: Shape parity
# ------------------------------------------------------------------
shape_ok = py_audit.shape == stata_audit.shape
audit_results.append(('1. Shape parity', shape_ok))
print(f'\n--- CHECK 1: Shape ---')
print(f'  Python: {py_audit.shape}   Stata: {stata_audit.shape}')
print(f'  {"PASS" if shape_ok else "FAIL"}')

# ------------------------------------------------------------------
# CHECK 2: Column names — exact set match
# ------------------------------------------------------------------
py_set = set(py_audit.columns)
st_set = set(stata_audit.columns)
col_set_ok = py_set == st_set
only_py_a = sorted(py_set - st_set)
only_st_a = sorted(st_set - py_set)
audit_results.append(('2. Column set match', col_set_ok))
print(f'\n--- CHECK 2: Column set ---')
print(f'  Match: {col_set_ok}')
if only_py_a: print(f'  Only in Python:  {only_py_a}')
if only_st_a: print(f'  Only in Stata:   {only_st_a}')

# ------------------------------------------------------------------
# CHECK 3: Column order — positional match
# ------------------------------------------------------------------
py_order = list(py_audit.columns)
st_order = list(stata_audit.columns)
order_ok = py_order == st_order
audit_results.append(('3. Column order', order_ok))
print(f'\n--- CHECK 3: Column order ---')
if order_ok:
    print(f'  All {len(py_order)} columns in identical order — PASS')
else:
    diverge = [(i, p, s) for i, (p, s) in enumerate(zip(py_order, st_order)) if p != s]
    print(f'  FAIL — {len(diverge)} position(s) differ')
    for i, p, s in diverge[:5]:
        print(f'    pos {i}: Python="{p}" vs Stata="{s}"')

# ------------------------------------------------------------------
# CHECK 4: Region column naming (critical: Stata lowercases import delimited without case(preserve))
# ------------------------------------------------------------------
region_cols_expected = {'region', 'division'}  # lowercase from Stata import delimited
region_ok = region_cols_expected.issubset(st_set) and region_cols_expected.issubset(py_set)
audit_results.append(('4. Region column names', region_ok))
print(f'\n--- CHECK 4: Region column naming ---')
print(f'  Expected lowercase "region","division" in both: {region_ok}')
for rc in ['region', 'division', 'Region', 'Division']:
    in_py = rc in py_set
    in_st = rc in st_set
    if in_py or in_st:
        print(f'    "{rc}": Python={in_py}, Stata={in_st}')

# ------------------------------------------------------------------
# CHECK 5: Key uniqueness (ppd_id, fy)
# ------------------------------------------------------------------
py_dup = py_audit.duplicated(subset=['ppd_id', 'fy'], keep=False).sum()
st_dup = stata_audit.duplicated(subset=['ppd_id', 'fy'], keep=False).sum()
dup_ok = py_dup == st_dup
audit_results.append(('5. Key uniqueness', dup_ok))
print(f'\n--- CHECK 5: Key uniqueness (ppd_id, fy) ---')
print(f'  Duplicate rows: Python={py_dup}, Stata={st_dup} — {"PASS" if dup_ok else "FAIL"}')

print(f'\n  First 5 checks complete. Continuing...')

        COMPREHENSIVE STEP 2 AUDIT

--- CHECK 1: Shape ---
  Python: (6268, 283)   Stata: (6268, 283)
  PASS

--- CHECK 2: Column set ---
  Match: True

--- CHECK 3: Column order ---
  All 283 columns in identical order — PASS

--- CHECK 4: Region column naming ---
  Expected lowercase "region","division" in both: True
    "region": Python=True, Stata=True
    "division": Python=True, Stata=True

--- CHECK 5: Key uniqueness (ppd_id, fy) ---
  Duplicate rows: Python=0, Stata=0 — PASS

  First 5 checks complete. Continuing...


In [22]:
# ------------------------------------------------------------------
# CHECK 6: Full null profile across ALL 283 columns
# Normalize Stata empty strings "" -> NaN before counting
# ------------------------------------------------------------------
print('\n--- CHECK 6: Full null profile (all 283 columns) ---')
null_mismatches = []
for c in stata_audit.columns:
    sn = stata_audit[c].isna().sum()
    pn = py_audit[c].isna().sum()
    # Normalize empty strings for string columns
    if pd.api.types.is_string_dtype(stata_audit[c]):
        sn += (stata_audit[c] == '').sum()
    if pd.api.types.is_string_dtype(py_audit[c]):
        pn += (py_audit[c] == '').sum()
    if pn != sn:
        null_mismatches.append((c, int(pn), int(sn)))

null_ok = len(null_mismatches) == 0
audit_results.append(('6. Full null profile', null_ok))
if null_ok:
    print(f'  All 283 columns have identical null counts — PASS')
else:
    print(f'  FAIL — {len(null_mismatches)} column(s) differ:')
    for c, pn, sn in null_mismatches[:10]:
        print(f'    {c}: Python={pn}, Stata={sn}')

# ------------------------------------------------------------------
# CHECK 7: Dtype category alignment (numeric vs string)
# ------------------------------------------------------------------
print('\n--- CHECK 7: Dtype category alignment ---')
dtype_issues = []
for c in stata_audit.columns:
    py_is_num = pd.api.types.is_numeric_dtype(py_audit[c])
    st_is_num = pd.api.types.is_numeric_dtype(stata_audit[c])
    if py_is_num != st_is_num:
        dtype_issues.append((c, str(py_audit[c].dtype), str(stata_audit[c].dtype)))

dtype_ok = len(dtype_issues) == 0
audit_results.append(('7. Dtype category alignment', dtype_ok))
if dtype_ok:
    print(f'  All columns agree on numeric vs string — PASS')
else:
    print(f'  FAIL — {len(dtype_issues)} column(s) have category mismatches:')
    for c, pd_, sd_ in dtype_issues[:10]:
        print(f'    {c}: Python={pd_}, Stata={sd_}')

# ------------------------------------------------------------------
# CHECK 8: Spot-check 6 specific rows by ppd_id + fy
# ------------------------------------------------------------------
print('\n--- CHECK 8: Spot-check specific rows ---')
spot_keys = [
    (1, 2001), (1, 2022),    # first plan, first and last year
    (100, 2010),              # middle plan, middle year
    (230, 2015),              # plan near end
    (16, 1939),               # unmatched row (no mapping match)
    (250, 2020),              # another plan
]

spot_ok = True
check_cols = ['PlanName', 'StateAbbrev', 'region', 'division', 'pub_id', 'pub_system_name']

for pid, fy_val in spot_keys:
    py_row = py_audit[(py_audit['ppd_id'] == pid) & (py_audit['fy'] == fy_val)]
    st_row = stata_audit[(stata_audit['ppd_id'] == pid) & (stata_audit['fy'] == fy_val)]
    
    if len(py_row) == 0 and len(st_row) == 0:
        print(f'  ppd_id={pid}, fy={fy_val}: not present in either — SKIP')
        continue
    elif len(py_row) != len(st_row):
        print(f'  ppd_id={pid}, fy={fy_val}: row count differs Py={len(py_row)} St={len(st_row)} — FAIL')
        spot_ok = False
        continue
    
    row_pass = True
    issues = []
    for c in check_cols:
        if c not in py_audit.columns:
            continue
        pv_ = py_row[c].values[0]
        sv_ = st_row[c].values[0]
        # Normalize NaN/empty
        pv_n = None if (pd.isna(pv_) or pv_ == '') else pv_
        sv_n = None if (pd.isna(sv_) or sv_ == '') else sv_
        if pv_n != sv_n:
            issues.append(f'{c}: Py={pv_!r} St={sv_!r}')
            row_pass = False
    
    status = 'PASS' if row_pass else 'FAIL'
    print(f'  ppd_id={pid}, fy={fy_val}: {status}', end='')
    if issues:
        print(f' [{"; ".join(issues)}]')
        spot_ok = False
    else:
        print()

audit_results.append(('8. Spot-check rows', spot_ok))

print(f'\n  Checks 6-8 complete. Continuing...')


--- CHECK 6: Full null profile (all 283 columns) ---
  All 283 columns have identical null counts — PASS

--- CHECK 7: Dtype category alignment ---
  All columns agree on numeric vs string — PASS

--- CHECK 8: Spot-check specific rows ---
  ppd_id=1, fy=2001: PASS
  ppd_id=1, fy=2022: PASS
  ppd_id=100, fy=2010: PASS
  ppd_id=230, fy=2015: PASS
  ppd_id=16, fy=1939: PASS
  ppd_id=250, fy=2020: not present in either — SKIP

  Checks 6-8 complete. Continuing...


In [23]:
# ------------------------------------------------------------------
# CHECK 9: Exhaustive cell-by-cell value comparison (reloaded from disk)
# ------------------------------------------------------------------
print('\n--- CHECK 9: Exhaustive value comparison (from saved files) ---')
all_cols = sorted(set(py_audit.columns) & set(stata_audit.columns))
py_s = py_audit[all_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
st_s = stata_audit[all_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)

# Normalize empty strings
for c in all_cols:
    if pd.api.types.is_string_dtype(st_s[c]):
        st_s[c] = st_s[c].replace('', np.nan)
    if pd.api.types.is_string_dtype(py_s[c]):
        py_s[c] = py_s[c].replace('', np.nan)

val_issues = {}
for c in all_cols:
    p = py_s[c]; s = st_s[c]
    nan_diff = int((p.isna() != s.isna()).sum())
    both_v = p.notna() & s.notna()
    if both_v.any():
        if pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(s):
            pn_ = pd.to_numeric(p[both_v], errors='coerce')
            sn_ = pd.to_numeric(s[both_v], errors='coerce')
            vd = int((~np.isclose(pn_, sn_, rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
        else:
            vd = int((p[both_v].astype(str) != s[both_v].astype(str)).sum())
    else:
        vd = 0
    if nan_diff + vd > 0:
        val_issues[c] = (nan_diff, vd)

val_ok = len(val_issues) == 0
audit_results.append(('9. Exhaustive value comparison', val_ok))
if val_ok:
    print(f'  All {len(all_cols)} columns × {len(py_s)} rows match — PASS')
else:
    print(f'  FAIL — {len(val_issues)} column(s):')
    for c, (nd, vd) in sorted(val_issues.items())[:15]:
        print(f'    {c}: nan_diff={nd}, val_diff={vd}')

# ------------------------------------------------------------------
# CHECK 10: Merge semantics verification
# Reload raw PPD and mapping to confirm keep(1 3) behavior
# ------------------------------------------------------------------
print('\n--- CHECK 10: Merge semantics (keep 1 3) ---')
ppd_raw = pd.read_csv(RAW / 'ppd-data-2023-07-28.csv', encoding='cp1252')
ppd_raw = ppd_raw.dropna(subset=['fy'])
mapping_raw = pd.read_stata(OUTPUT_STATA / 'mapping_table_fy_final.dta')
mapping_raw['fy'] = mapping_raw['fy'].astype(int)
mapping_raw['ppd_id'] = mapping_raw['ppd_id'].astype(int)
ppd_raw['ppd_id'] = ppd_raw['ppd_id'].astype(int)
ppd_raw['fy'] = ppd_raw['fy'].astype(int)

# m:1 merge with indicator
test_merge = ppd_raw.merge(mapping_raw[['ppd_id', 'fy']], on=['ppd_id', 'fy'], how='outer', indicator=True)
master_only = (test_merge['_merge'] == 'left_only').sum()
both = (test_merge['_merge'] == 'both').sum()
using_only = (test_merge['_merge'] == 'right_only').sum()

# keep(1 3) means final should have master_only + both = original PPD count
expected_rows = master_only + both
merge_ok = expected_rows == len(py_audit) == len(stata_audit)
audit_results.append(('10. Merge keep(1 3) semantics', merge_ok))
print(f'  PPD master-only: {master_only}')
print(f'  Matched (both): {both}')
print(f'  Mapping using-only (dropped): {using_only}')
print(f'  Expected rows (1+3): {expected_rows}')
print(f'  Python rows: {len(py_audit)}, Stata rows: {len(stata_audit)}')
print(f'  {"PASS" if merge_ok else "FAIL"}')

# ------------------------------------------------------------------
# FINAL AUDIT SUMMARY
# ------------------------------------------------------------------
print('\n' + '=' * 60)
print('        AUDIT SUMMARY')
print('=' * 60)
all_ok = True
for name, passed in audit_results:
    icon = 'PASS' if passed else 'FAIL'
    print(f'  [{icon}] {name}')
    if not passed:
        all_ok = False
print('=' * 60)
if all_ok:
    print('  ✅  ALL 10 AUDIT CHECKS PASSED — EXACT PARITY CONFIRMED')
else:
    print('  ❌  SOME CHECKS FAILED — REVIEW ISSUES ABOVE')
print('=' * 60)


--- CHECK 9: Exhaustive value comparison (from saved files) ---
  All 283 columns × 6268 rows match — PASS

--- CHECK 10: Merge semantics (keep 1 3) ---
  PPD master-only: 1252
  Matched (both): 5016
  Mapping using-only (dropped): 228
  Expected rows (1+3): 6268
  Python rows: 6268, Stata rows: 6268
  PASS

        AUDIT SUMMARY
  [PASS] 1. Shape parity
  [PASS] 2. Column set match
  [PASS] 3. Column order
  [PASS] 4. Region column names
  [PASS] 5. Key uniqueness
  [PASS] 6. Full null profile
  [PASS] 7. Dtype category alignment
  [PASS] 8. Spot-check rows
  [PASS] 9. Exhaustive value comparison
  [PASS] 10. Merge keep(1 3) semantics
  ✅  ALL 10 AUDIT CHECKS PASSED — EXACT PARITY CONFIRMED


## 9. Joint Audit — Step 1 + Step 2 Together

Tests the full chain:
1. **Step 1 standalone** — `output_python/mapping_table_fy_final.dta` vs `output/mapping_table_fy_final.dta`
2. **Step 2 standalone** — `output_python/ppd_plan_level_raw.dta` vs `output/ppd_plan_level_raw.dta`
3. **Chained pipeline** — Re-run Step 2 merges using Python Step 1 output as mapping input, compare to Stata baseline
4. **Cross-step consistency** — Verify mapping columns flow correctly from Step 1 into Step 2

In [24]:
print('=' * 65)
print('     JOINT AUDIT: Step 1 + Step 2 Pipeline')
print('=' * 65)

joint = []

# ======================================================================
# PART A: Step 1 standalone parity (Python mapping vs Stata mapping)
# ======================================================================
print('\n' + '-' * 65)
print(' PART A: Step 1 — mapping_table_fy_final')
print('-' * 65)

py_map = pd.read_stata(OUTPUT_PY / 'mapping_table_fy_final.dta')
st_map = pd.read_stata(OUTPUT_STATA / 'mapping_table_fy_final.dta')

# A1: Shape
a1 = py_map.shape == st_map.shape
joint.append(('A1. Step 1 shape', a1))
print(f'A1. Shape: Py={py_map.shape} St={st_map.shape} — {"PASS" if a1 else "FAIL"}')

# A2: Column set + order
a2 = list(py_map.columns) == list(st_map.columns)
joint.append(('A2. Step 1 column order', a2))
print(f'A2. Column order: {"PASS" if a2 else "FAIL"}')

# A3: Exhaustive value comparison
map_cols = sorted(set(py_map.columns) & set(st_map.columns))
py_m = py_map[map_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
st_m = st_map[map_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
for c in map_cols:
    if pd.api.types.is_string_dtype(st_m[c]):
        st_m[c] = st_m[c].replace('', np.nan)
    if pd.api.types.is_string_dtype(py_m[c]):
        py_m[c] = py_m[c].replace('', np.nan)

map_diffs = 0
for c in map_cols:
    p, s = py_m[c], st_m[c]
    nd = int((p.isna() != s.isna()).sum())
    bv = p.notna() & s.notna()
    if bv.any() and (pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(s)):
        vd = int((~np.isclose(pd.to_numeric(p[bv], errors='coerce'),
                              pd.to_numeric(s[bv], errors='coerce'),
                              rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
    elif bv.any():
        vd = int((p[bv].astype(str) != s[bv].astype(str)).sum())
    else:
        vd = 0
    map_diffs += nd + vd

a3 = map_diffs == 0
joint.append(('A3. Step 1 value parity', a3))
print(f'A3. Value comparison ({len(map_cols)} cols × {len(py_m)} rows): {map_diffs} mismatches — {"PASS" if a3 else "FAIL"}')

# ======================================================================
# PART B: Step 2 standalone parity (Python PPD raw vs Stata PPD raw)
# ======================================================================
print('\n' + '-' * 65)
print(' PART B: Step 2 — ppd_plan_level_raw')
print('-' * 65)

py_ppd = pd.read_stata(OUTPUT_PY / 'ppd_plan_level_raw.dta')
st_ppd = pd.read_stata(OUTPUT_STATA / 'ppd_plan_level_raw.dta')

# B1: Shape
b1 = py_ppd.shape == st_ppd.shape
joint.append(('B1. Step 2 shape', b1))
print(f'B1. Shape: Py={py_ppd.shape} St={st_ppd.shape} — {"PASS" if b1 else "FAIL"}')

# B2: Column set + order
b2 = list(py_ppd.columns) == list(st_ppd.columns)
joint.append(('B2. Step 2 column order', b2))
print(f'B2. Column order: {"PASS" if b2 else "FAIL"}')

# B3: Exhaustive value comparison
ppd_cols = sorted(set(py_ppd.columns) & set(st_ppd.columns))
py_p = py_ppd[ppd_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
st_p = st_ppd[ppd_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
for c in ppd_cols:
    if pd.api.types.is_string_dtype(st_p[c]):
        st_p[c] = st_p[c].replace('', np.nan)
    if pd.api.types.is_string_dtype(py_p[c]):
        py_p[c] = py_p[c].replace('', np.nan)

ppd_diffs = 0
for c in ppd_cols:
    p, s = py_p[c], st_p[c]
    nd = int((p.isna() != s.isna()).sum())
    bv = p.notna() & s.notna()
    if bv.any() and (pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(s)):
        vd = int((~np.isclose(pd.to_numeric(p[bv], errors='coerce'),
                              pd.to_numeric(s[bv], errors='coerce'),
                              rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
    elif bv.any():
        vd = int((p[bv].astype(str) != s[bv].astype(str)).sum())
    else:
        vd = 0
    ppd_diffs += nd + vd

b3 = ppd_diffs == 0
joint.append(('B3. Step 2 value parity', b3))
print(f'B3. Value comparison ({len(ppd_cols)} cols × {len(py_p)} rows): {ppd_diffs} mismatches — {"PASS" if b3 else "FAIL"}')

     JOINT AUDIT: Step 1 + Step 2 Pipeline

-----------------------------------------------------------------
 PART A: Step 1 — mapping_table_fy_final
-----------------------------------------------------------------
A1. Shape: Py=(5244, 15) St=(5244, 15) — PASS
A2. Column order: PASS
A3. Value comparison (15 cols × 5244 rows): 0 mismatches — PASS

-----------------------------------------------------------------
 PART B: Step 2 — ppd_plan_level_raw
-----------------------------------------------------------------
B1. Shape: Py=(6268, 283) St=(6268, 283) — PASS
B2. Column order: PASS
B3. Value comparison (283 cols × 6268 rows): 0 mismatches — PASS


In [25]:
# ======================================================================
# PART C: Chained pipeline — Re-run Step 2 using Python Step 1 output
#         Compare result to Stata Step 2 baseline
# ======================================================================
print('-' * 65)
print(' PART C: Chained — Python Step 1 output → Step 2 → vs Stata')
print('-' * 65)

# Replay Step 2 logic from scratch using Python mapping
regions_c = pd.read_csv(RAW / 'us_census_bureau_regions_and_divisions.csv')
regions_c.columns = regions_c.columns.str.lower().str.replace(' ', '', regex=False)
regions_c = regions_c.rename(columns={'statecode': 'StateAbbrev'}).drop(columns=['state'])

ppd_c = pd.read_csv(RAW / 'ppd-data-2023-07-28.csv', encoding='cp1252')
ppd_c = ppd_c.dropna(subset=['fy'])
ppd_c['fy'] = ppd_c['fy'].astype(int)
ppd_c['ppd_id'] = ppd_c['ppd_id'].astype(int)

# KEY DIFFERENCE: use Python Step 1 output instead of Stata baseline
mapping_py = pd.read_stata(OUTPUT_PY / 'mapping_table_fy_final.dta')
mapping_py['fy'] = mapping_py['fy'].astype(int)
mapping_py['ppd_id'] = mapping_py['ppd_id'].astype(int)

ppd_c = ppd_c.merge(mapping_py, on=['ppd_id', 'fy'], how='left')
ppd_c = ppd_c.merge(regions_c, on='StateAbbrev', how='left')
ppd_c = ppd_c.sort_values(['ppd_id', 'fy']).reset_index(drop=True)

print(f'Chained result: {ppd_c.shape[0]} rows × {ppd_c.shape[1]} cols')

# C1: Shape vs Stata baseline
c1 = ppd_c.shape == st_ppd.shape
joint.append(('C1. Chained shape', c1))
print(f'C1. Shape vs Stata: Py_chain={ppd_c.shape} Stata={st_ppd.shape} — {"PASS" if c1 else "FAIL"}')

# C2: Column set
c2 = set(ppd_c.columns) == set(st_ppd.columns)
joint.append(('C2. Chained column set', c2))
print(f'C2. Column set: {"PASS" if c2 else "FAIL"}')
if not c2:
    only_chain = sorted(set(ppd_c.columns) - set(st_ppd.columns))
    only_st = sorted(set(st_ppd.columns) - set(ppd_c.columns))
    if only_chain: print(f'    Only in chained: {only_chain}')
    if only_st: print(f'    Only in Stata:   {only_st}')

# C3: Exhaustive value comparison
chain_cols = sorted(set(ppd_c.columns) & set(st_ppd.columns))
py_ch = ppd_c[chain_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
st_ch = st_ppd[chain_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)

for c in chain_cols:
    if pd.api.types.is_string_dtype(st_ch[c]):
        st_ch[c] = st_ch[c].replace('', np.nan)
    if pd.api.types.is_string_dtype(py_ch[c]):
        py_ch[c] = py_ch[c].replace('', np.nan)

chain_diffs = 0
chain_detail = {}
for c in chain_cols:
    p, s = py_ch[c], st_ch[c]
    nd = int((p.isna() != s.isna()).sum())
    bv = p.notna() & s.notna()
    if bv.any() and (pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(s)):
        vd = int((~np.isclose(pd.to_numeric(p[bv], errors='coerce'),
                              pd.to_numeric(s[bv], errors='coerce'),
                              rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
    elif bv.any():
        vd = int((p[bv].astype(str) != s[bv].astype(str)).sum())
    else:
        vd = 0
    if nd + vd > 0:
        chain_detail[c] = (nd, vd)
    chain_diffs += nd + vd

c3 = chain_diffs == 0
joint.append(('C3. Chained value parity', c3))
print(f'C3. Value comparison ({len(chain_cols)} cols × {len(py_ch)} rows): {chain_diffs} mismatches — {"PASS" if c3 else "FAIL"}')
if chain_detail:
    for c_name, (nd, vd) in sorted(chain_detail.items())[:10]:
        print(f'    {c_name}: nan_diff={nd}, val_diff={vd}')

# ======================================================================
# PART D: Cross-step consistency — mapping columns flow correctly
# ======================================================================
print('\n' + '-' * 65)
print(' PART D: Cross-step — mapping columns in Step 2 match Step 1')
print('-' * 65)

# The mapping columns that Step 1 produces and Step 2 merges in
mapping_cols = [c for c in py_map.columns if c not in ('ppd_id', 'fy')]
print(f'Mapping columns to verify: {mapping_cols}')

# For matched rows in Step 2, the mapping column values should come from Step 1
py_ppd_s = py_ppd.sort_values(['ppd_id', 'fy']).reset_index(drop=True)
st_ppd_s = st_ppd.sort_values(['ppd_id', 'fy']).reset_index(drop=True)

cross_ok = True
cross_issues = []

for mc in mapping_cols:
    if mc not in py_ppd_s.columns or mc not in st_ppd_s.columns:
        cross_issues.append(f'{mc}: missing from one dataset')
        cross_ok = False
        continue
    
    p_col = py_ppd_s[mc].copy()
    s_col = st_ppd_s[mc].copy()
    
    # Normalize empty strings
    if pd.api.types.is_string_dtype(p_col):
        p_col = p_col.replace('', np.nan)
    if pd.api.types.is_string_dtype(s_col):
        s_col = s_col.replace('', np.nan)
    
    # Compare
    nan_d = int((p_col.isna() != s_col.isna()).sum())
    bv = p_col.notna() & s_col.notna()
    if bv.any() and (pd.api.types.is_numeric_dtype(p_col) or pd.api.types.is_numeric_dtype(s_col)):
        vd = int((~np.isclose(pd.to_numeric(p_col[bv], errors='coerce'),
                              pd.to_numeric(s_col[bv], errors='coerce'),
                              rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
    elif bv.any():
        vd = int((p_col[bv].astype(str) != s_col[bv].astype(str)).sum())
    else:
        vd = 0
    
    status_mc = 'PASS' if (nan_d + vd == 0) else 'FAIL'
    print(f'  {mc}: nan_diff={nan_d}, val_diff={vd} — {status_mc}')
    if nan_d + vd > 0:
        cross_ok = False
        cross_issues.append(f'{mc}: nan_diff={nan_d}, val_diff={vd}')

joint.append(('D. Cross-step mapping columns', cross_ok))

# ======================================================================
# PART E: Verify no data leakage / phantom columns
# ======================================================================
print('\n' + '-' * 65)
print(' PART E: No phantom columns from merges')
print('-' * 65)

# Expected columns = original PPD CSV columns + mapping columns + region columns
ppd_raw_cols = set(pd.read_csv(RAW / 'ppd-data-2023-07-28.csv', encoding='cp1252', nrows=0).columns)
map_added = set(py_map.columns) - {'ppd_id', 'fy'}  # merged in
region_added = set(regions_c.columns) - {'StateAbbrev'}  # merged in
expected_cols = ppd_raw_cols | map_added | region_added

actual_cols = set(py_ppd.columns)
phantom = actual_cols - expected_cols
missing = expected_cols - actual_cols

e_ok = len(phantom) == 0 and len(missing) == 0
joint.append(('E. No phantom/missing columns', e_ok))
print(f'  Expected columns: {len(expected_cols)}')
print(f'  Actual columns:   {len(actual_cols)}')
if phantom: print(f'  Phantom (unexpected): {sorted(phantom)}')
if missing: print(f'  Missing (expected):   {sorted(missing)}')
print(f'  {"PASS" if e_ok else "FAIL"}')

# ======================================================================
# FINAL JOINT AUDIT SUMMARY
# ======================================================================
print('\n' + '=' * 65)
print('     JOINT AUDIT SUMMARY')
print('=' * 65)
all_joint_ok = True
for name, passed in joint:
    icon = 'PASS' if passed else 'FAIL'
    print(f'  [{icon}] {name}')
    if not passed:
        all_joint_ok = False
print('=' * 65)
if all_joint_ok:
    print('  ✅  ALL JOINT AUDIT CHECKS PASSED')
    print('     Step 1 → Step 2 pipeline produces exact parity')
    print('     with the original Stata output files.')
else:
    print('  ❌  SOME CHECKS FAILED — REVIEW ABOVE')
print('=' * 65)

-----------------------------------------------------------------
 PART C: Chained — Python Step 1 output → Step 2 → vs Stata
-----------------------------------------------------------------
Chained result: 6268 rows × 283 cols
C1. Shape vs Stata: Py_chain=(6268, 283) Stata=(6268, 283) — PASS
C2. Column set: PASS
C3. Value comparison (283 cols × 6268 rows): 0 mismatches — PASS

-----------------------------------------------------------------
 PART D: Cross-step — mapping columns in Step 2 match Step 1
-----------------------------------------------------------------
Mapping columns to verify: ['ppd_name', 'pub_id', 'pub_system_name', 'preqin_id', 'preqin_name', 'tr13fid1', 'tr13fname1', 'tr13fid2', 'tr13fname2', 'tr13fid3', 'tr13fname3', 'tr13fid4', 'tr13fname4']
  ppd_name: nan_diff=0, val_diff=0 — PASS
  pub_id: nan_diff=0, val_diff=0 — PASS
  pub_system_name: nan_diff=0, val_diff=0 — PASS
  preqin_id: nan_diff=0, val_diff=0 — PASS
  preqin_name: nan_diff=0, val_diff=0 — PASS
  tr1

In [26]:
# Compact summary of joint audit results
print('JOINT AUDIT RESULTS:')
for name, passed in joint:
    print(f'  [{"PASS" if passed else "FAIL"}] {name}')
all_j = all(p for _, p in joint)
print(f'\nOverall: {"ALL PASSED" if all_j else "SOME FAILED"}')

JOINT AUDIT RESULTS:
  [PASS] A1. Step 1 shape
  [PASS] A2. Step 1 column order
  [PASS] A3. Step 1 value parity
  [PASS] B1. Step 2 shape
  [PASS] B2. Step 2 column order
  [PASS] B3. Step 2 value parity
  [PASS] C1. Chained shape
  [PASS] C2. Chained column set
  [PASS] C3. Chained value parity
  [PASS] D. Cross-step mapping columns
  [PASS] E. No phantom/missing columns

Overall: ALL PASSED
